# Aula 1, Regressão

Notebook executável que acompanha a aula [01-regressao.md](../../lessons/modulo-02-fundamentos-ml/01-regressao.md).

## O que vamos fazer aqui

Vamos prever a nota de um aluno a partir das horas de estudo, que é um problema de
regressão, porque a resposta é um número contínuo. Para isso, implementamos uma
regressão linear do zero, treinada por gradiente descendente, e acompanhamos o modelo
aprender a melhor reta.

O núcleo usa apenas numpy. As partes de gráfico e de comparação com o scikit-learn
são opcionais e degradam de forma graciosa se a biblioteca não estiver instalada.

## Gerando os dados

Como não temos um conjunto real em mãos, criamos dados sintéticos com uma tendência
clara, mais estudo tende a render nota mais alta, somada a um ruído que imita a
variação natural da vida real. A relação verdadeira é nota = 1.2 × horas + 3.

In [ ]:
import numpy as np

# Dados sintéticos: horas de estudo (x) e nota final (y), com ruído.
rng = np.random.default_rng(0)
horas = rng.uniform(0, 10, size=80)
notas = 1.2 * horas + 3 + rng.normal(0, 1.0, size=80)
print('exemplos:', len(horas))
print('primeiras horas:', np.round(horas[:5], 2))
print('primeiras notas:', np.round(notas[:5], 2))

Temos 80 pares de horas e notas. O objetivo do modelo é descobrir, só a
partir desses pares, a relação que usamos para gerá-los, sem que a gente conte qual
é.

## Regressão linear do zero

O gradiente descendente ajusta os parâmetros w e b dando pequenos passos na direção
que mais reduz o erro quadrático médio. A cada iteração, calculamos o gradiente e
atualizamos os parâmetros. A taxa de aprendizado controla o tamanho do passo.

In [ ]:
def treinar_regressao(x, y, taxa=0.01, iteracoes=2000):
    """Ajusta w e b por gradiente descendente, minimizando o MSE."""
    w, b = 0.0, 0.0
    m = len(x)
    for _ in range(iteracoes):
        y_prev = w * x + b
        erro = y_prev - y
        grad_w = (2 / m) * np.sum(erro * x)
        grad_b = (2 / m) * np.sum(erro)
        w -= taxa * grad_w
        b -= taxa * grad_b
    return w, b


w, b = treinar_regressao(horas, notas)
print(f'Modelo aprendido: nota = {w:.3f} * horas + {b:.3f}')

mse = np.mean((w * horas + b - notas) ** 2)
print(f'MSE no treino: {mse:.3f}')
print(f'Nota prevista para 6 horas: {w * 6 + b:.2f}')

Os valores de w e b aprendidos ficam próximos da relação verdadeira (1.2 e
3), sem serem idênticos por causa do ruído. Isso é o que se espera de um bom modelo:
ele captura a tendência sem decorar o ruído, tema que aprofundamos na aula de
overfitting.

## Visualização e comparação opcionais

Esta célula desenha os dados e a reta aprendida, e compara os parâmetros com os do
scikit-learn. As duas partes são opcionais, então o notebook roda mesmo sem
matplotlib ou scikit-learn instalados.

In [ ]:
# Gráfico opcional: roda se o matplotlib estiver instalado.
try:
    import matplotlib.pyplot as plt

    plt.scatter(horas, notas, alpha=0.6, label='dados')
    xs = np.linspace(0, 10, 100)
    plt.plot(xs, w * xs + b, color='red', label='reta aprendida')
    plt.xlabel('horas de estudo')
    plt.ylabel('nota')
    plt.legend()
    plt.show()
except ImportError:
    print('matplotlib não instalado, pulando o gráfico (veja docs/setup.md).')

# Comparação opcional com o scikit-learn.
try:
    from sklearn.linear_model import LinearRegression

    modelo_sk = LinearRegression().fit(horas.reshape(-1, 1), notas)
    print(f'scikit-learn: w = {modelo_sk.coef_[0]:.3f}, b = {modelo_sk.intercept_:.3f}')
    print(f'nosso modelo: w = {w:.3f}, b = {b:.3f}')
except ImportError:
    print('scikit-learn não instalado, pulando a comparação (veja docs/setup.md).')

Se o scikit-learn estiver disponível, os parâmetros dele devem bater quase
exatamente com os nossos, o que é uma boa validação da implementação feita à mão.

Para o projeto da aula, gere um segundo conjunto com a mesma regra, meça o erro do
modelo nesses dados novos que ele não viu, e compare com o erro de treino. Essa
diferença é a semente das aulas de overfitting e validação.